In [2]:
print("1- Importations et chargement des données \n")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import make_pipeline

# Chargement du train set prétraité
train_df = pd.read_csv("../data/processed/train_preprocessed.csv")

print("Dimensions du train set :", train_df.shape)
print("Distribution de la cible (stroke) :")
print(train_df["stroke"].value_counts(normalize=True) * 100)

1- Importations et chargement des données 

Dimensions du train set : (4087, 17)
Distribution de la cible (stroke) :
stroke
0    95.130903
1     4.869097
Name: proportion, dtype: float64


In [3]:
print("2- Séparation des caractéristiques et de la cible\n ")


X_train = train_df.drop(columns=["stroke"])
y_train = train_df["stroke"]

print("Variables explicatives :", list(X_train.columns))
print("Nombre de caractéristiques :", X_train.shape[1])

2- Séparation des caractéristiques et de la cible
 
Variables explicatives : ['gender', 'age', 'hypertension', 'heart_disease', 'ever_married', 'Residence_type', 'avg_glucose_level', 'bmi', 'bmi_missing', 'work_type_Never_worked', 'work_type_Private', 'work_type_Self-employed', 'work_type_children', 'smoking_status_formerly smoked', 'smoking_status_never smoked', 'smoking_status_smokes']
Nombre de caractéristiques : 16


In [4]:
print("3- Configuration de la validation croisée et des métriques\n")

# Utilisation d'une validation croisée stratifiée pour respecter le déséquilibre de classe
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Définition des métriques à évaluer
scoring = ['recall', 'precision', 'average_precision', 'roc_auc']

3- Configuration de la validation croisée et des métriques



In [5]:
print("4- Definition et evaluation des modeles\n")

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

# SMOTE is placed in a pipeline so resampling happens only on each fold's training data.
# Tree-based estimators are regularized to limit overfitting.
models = {
    # Logistic Regression: no correction, class weighting, and SMOTE.
    "LogisticRegression (aucune)": LogisticRegression(random_state=42),
    "LogisticRegression (class_weight)": LogisticRegression(
        class_weight="balanced",
        random_state=42,
    ),
    "LogisticRegression (SMOTE)": make_pipeline(
        SMOTE(random_state=42),
        LogisticRegression(random_state=42),
    ),

    # KNN does not support class_weight; keep its applicable strategies.
    "KNN (aucun)": KNeighborsClassifier(),
    "KNN (SMOTE)": make_pipeline(
        SMOTE(random_state=42),
        KNeighborsClassifier(),
    ),

    # Decision Tree regularized with a maximum depth.
    "DecisionTree (aucune)": DecisionTreeClassifier(
        max_depth=4,
        random_state=42,
    ),
    "DecisionTree (class_weight)": DecisionTreeClassifier(
        max_depth=4,
        class_weight="balanced",
        random_state=42,
    ),
    "DecisionTree (SMOTE)": make_pipeline(
        SMOTE(random_state=42),
        DecisionTreeClassifier(max_depth=4, random_state=42),
    ),

    # Regularized Random Forest to limit overfitting.
    "RandomForest (aucune)": RandomForestClassifier(
        max_depth=4,
        min_samples_leaf=10,
        n_estimators=100,
        random_state=42,
    ),
    "RandomForest (class_weight)": RandomForestClassifier(
        max_depth=4,
        min_samples_leaf=10,
        n_estimators=100,
        class_weight="balanced",
        random_state=42,
    ),
    "RandomForest (SMOTE)": make_pipeline(
        SMOTE(random_state=42),
        RandomForestClassifier(
            max_depth=4,
            min_samples_leaf=10,
            n_estimators=100,
            random_state=42,
        ),
    ),

    # Regularized XGBoost; scale_pos_weight adjusts the positive class weight.
    "XGBoost (aucune)": XGBClassifier(
        max_depth=3,
        learning_rate=0.05,
        n_estimators=100,
        random_state=42,
    ),
    "XGBoost (class_weight)": XGBClassifier(
        max_depth=3,
        learning_rate=0.05,
        n_estimators=100,
        scale_pos_weight=(len(y_train) - y_train.sum()) / y_train.sum(),
        random_state=42,
    ),
    "XGBoost (SMOTE)": make_pipeline(
        SMOTE(random_state=42),
        XGBClassifier(
            max_depth=3,
            learning_rate=0.05,
            n_estimators=100,
            random_state=42,
        ),
    ),
}

results = []

for name, model in models.items():
    cv_results = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
    results.append({
        "Mod\u00e8le": name,
        "Recall": cv_results["test_recall"].mean(),
        "Precision": cv_results["test_precision"].mean(),
        "PR-AUC": cv_results["test_average_precision"].mean(),
        "ROC-AUC": cv_results["test_roc_auc"].mean(),
    })

results_df = pd.DataFrame(results)
# Sort by PR-AUC descending to compare models on the minority class.
results_df = results_df.sort_values(
    by="PR-AUC",
    ascending=False,
).reset_index(drop=True)
display(results_df)

4- Definition et evaluation des modeles



c:\Users\pc\Desktop\AVC-D-tection-Syst-me\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\pc\Desktop\AVC-D-tection-Syst-me\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\pc\Desktop\AVC-D-tection-Syst-me\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", 

,Modèle,Recall,Precision,PR-AUC,ROC-AUC
0,RandomForest (aucune),0.000000,0.000000,0.249925,0.851289
1,LogisticRegression (aucune),0.020128,0.266667,0.227259,0.852870
2,XGBoost (aucune),0.015000,0.500000,0.226421,0.840862
3,LogisticRegression (class_weight),0.799231,0.142029,0.224562,0.853658
4,XGBoost (class_weight),0.784231,0.124089,0.217606,0.837178
5,RandomForest (class_weight),0.884487,0.112537,0.216084,0.844710
6,DecisionTree (aucune),0.075513,0.488645,0.200077,0.826199
7,RandomForest (SMOTE),0.879487,0.111426,0.191828,0.822926
8,LogisticRegression (SMOTE),0.683590,0.130113,0.182464,0.816256
9,DecisionTree (class_weight),0.783718,0.126841,0.181328,0.819975


In [6]:
print("5- Selection des meilleurs modeles pour l'evaluation finale\n")

# Keep the transparent weighted Logistic Regression as the clinical baseline.
CHOICES = [("LogisticRegression", "class_weight")]

# Select the best PR-AUC strategy for each ensemble family.
for family in ("RandomForest", "XGBoost"):
    family_results = results_df[
        results_df["Mod\u00e8le"].str.startswith(f"{family} (")
    ]
    best_model_name = family_results.loc[
        family_results["PR-AUC"].idxmax(),
        "Mod\u00e8le",
    ]
    model_name, strategy = best_model_name.rsplit(" (", 1)
    CHOICES.append((model_name, strategy.rstrip(")")))

print("Modeles selectionnes pour l'evaluation finale (04_evaluation) :")
for model, strategy in CHOICES:
    print(f"- {model} avec la strategie {strategy}")

print(
    "\nImportant : pour Random Forest et XGBoost, 04_evaluation.ipynb devra "
    "obligatoirement optimiser le seuil de decision a partir de la courbe "
    "Precision-Recall. Le seuil par defaut de 0.50 est inadapte aux donnees "
    "desequilibrees ; une plage initiale de 0.15 a 0.30 pourra etre exploree "
    "pour augmenter le Recall."
)

5- Selection des meilleurs modeles pour l'evaluation finale

Modeles selectionnes pour l'evaluation finale (04_evaluation) :
- LogisticRegression avec la strategie class_weight
- RandomForest avec la strategie aucune
- XGBoost avec la strategie aucune

Important : pour Random Forest et XGBoost, 04_evaluation.ipynb devra obligatoirement optimiser le seuil de decision a partir de la courbe Precision-Recall. Le seuil par defaut de 0.50 est inadapte aux donnees desequilibrees ; une plage initiale de 0.15 a 0.30 pourra etre exploree pour augmenter le Recall.
